# 03d · Shallow survival tree: time to platinum, transferred to NEPC / AVPC

A depth-1 to depth-3 survival tree on the predictive checklist's items. Each leaf is a risk group defined by at most
three lab or age cutoffs, with a shrunken event rate (the risk score) and a Kaplan–Meier curve. The split rule is
rpart's `method = "exp"`, and the model code is a byte-identical copy of the federated app
(`rhino_scripts/feature_set_survival_tree`). Method and outputs:
`COMPASS/multivariable_models/survival_tree/README.md`.

1. **Local cross-fit (DFCI).** Each outer fold's tree is selected (inner 5-fold CV over depth × `min_leaf`) and fit
   on the other folds' **platinum** rows only. It is then applied to the held-out fold's platinum, NEPC and AVPC
   rows. One patient-level fold split is shared by all landmarks and endpoints. NEPC and AVPC labels never enter
   fitting or selection, so the NEPC and AVPC results are the platinum tree's transfer.
2. **Federated tree (no DFCI).** The frozen `survival_tree` bundle refit without Dana-Farber is scored once on the
   same DFCI frames with `fet.evaluate_transfer`, using the same seed and bootstrap. The tree is not chosen on DFCI.

`both` may split on every lab and age item. `baseline` splits on age only and is the paired reference for Δ.

## Configuration

In [ ]:
import json
import os
import sys
from pathlib import Path

HERE = Path.cwd()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / "COMPASS" / "multivariable_models" / "survival_tree" / "local_tree.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository root or COMPASS/notebooks")
for _p in (PROJECT_ROOT, PROJECT_ROOT / "COMPASS" / "federated_models"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter

from COMPASS.multivariable_models.survival_tree import local_tree as lt
from survival_common import federated_endpoint_transfer as fet
from survival_common.federated_inference import load_bundle, merge_bundles
import federated_clinical_comparison as fcc   # Gleason / sequencing subsets (as 04)

DATA_ROOT   = Path(os.environ.get("COMPASS_DATA_ROOT", "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS"))
RHINO_ROOT  = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))
DATA_PATH   = Path(os.environ.get("COMPASS_ADT_DATA", str(DATA_ROOT / "longitudinal_prediction_data_adt.csv")))
LAB_MAPPING = Path(os.environ.get("COMPASS_LAB_MAPPING", str(fet.DEFAULT_LAB_MAPPING)))
OUTPUT_DIR  = Path(os.environ.get("COMPASS_SURVIVAL_TREE_ROOT", str(DATA_ROOT / "survival_tree_adt")))
LOCAL_DIR   = OUTPUT_DIR / "local_cross_fit"
TRANSFER_DIR = OUTPUT_DIR / "federated_transfer_no_DFCI"

# >>> Edit <<<
CFG = lt.TreeConfig(
    landmarks=(0, 90, 180),
    max_depths=(1, 2, 3),
    min_leaf_fracs=(0.02, 0.05, 0.10),
    n_folds=5, n_inner_folds=5,
    exclusion="none",          # "pre_anchor_castrate" to match noprecastrate bundles
    n_bootstrap=200, seed=0,
    horizons_days=(365, 1825),
)
ENDPOINTS = lt.ENDPOINTS        # platinum (training) + nepc, avpc (transfer)
FEDERATED_TREE_FILE = "survival_tree_model_*.json"   # the federated app's bundle name, any analysis label

def is_no_dfci(path, root):
    return any(part == "no_DFCI" or part.endswith("_no_DFCI") for part in path.relative_to(root).parts)

FEDERATED_TREE_PATHS = sorted(p for p in RHINO_ROOT.rglob(FEDERATED_TREE_FILE)
                              if is_no_dfci(p, RHINO_ROOT) and OUTPUT_DIR not in p.parents) if RHINO_ROOT.is_dir() else []
want_noprecastrate = CFG.exclusion == "pre_anchor_castrate"
FEDERATED_TREE_PATHS = [p for p in FEDERATED_TREE_PATHS if ("noprecastrate" in p.name) == want_noprecastrate]
print(f"data: {DATA_PATH}\noutputs: {OUTPUT_DIR}")
print(f"federated no-DFCI tree bundle(s) under {RHINO_ROOT}: {len(FEDERATED_TREE_PATHS)}")
for p in FEDERATED_TREE_PATHS:
    print("  ", p.relative_to(RHINO_ROOT))

## Local cohort

In [ ]:
local = fet.read_local_frame(DATA_PATH, ENDPOINTS)
lab_mapping = fet.load_lab_mapping(LAB_MAPPING)

pt = local.drop_duplicates(fet.LOCAL_ID_COL)
print(f"{len(local):,} rows, {len(pt):,} patients from {DATA_PATH}")
pd.DataFrame({
    ep: {"events": int(pt[ind].fillna(0).astype(int).sum()),
         "dated events": int((pt[ind].fillna(0).astype(int).eq(1) & pt[dt].notna()).sum())}
    for ep, (ind, dt) in fet.ENDPOINT_SOURCES.items() if ep in ENDPOINTS
}).T

## Local cross-fit

This runs 2 configs × 3 `min_leaf` fractions × 5 inner folds per landmark and outer fold, plus a full-cohort refit
(`fold = "all"`). The patient-level file is written under `OUTPUT_DIR` (protected data root).

In [ ]:
result = lt.run(local, lab_mapping, CFG)
lt.write_outputs(result, LOCAL_DIR)
with open(LOCAL_DIR / "run.json", "w") as fh:
    json.dump({"data": str(DATA_PATH), "lab_mapping": str(LAB_MAPPING), "config": CFG.to_dict(),
               "prep_stats": result["prep_stats"]}, fh, indent=2, default=str)
metrics = result["metrics"]

### Held-out performance (out of fold)

C and mean AUC(t) are given with 95% bootstrap CIs. Δ is `both` minus the age-only tree, paired on the same patients.
The bootstrap is conditional on the fold trees.

In [ ]:
def fmt(est, lo, hi):
    return "" if pd.isna(est) else (f"{est:.3f}" if pd.isna(lo) else f"{est:.3f} [{lo:.3f}, {hi:.3f}]")

def summary_table(m):
    m = m.copy()
    for col in ("c_index_lo", "c_index_hi", "mean_auc_lo", "mean_auc_hi", "delta_auc_vs_baseline",
                "delta_auc_lo", "delta_auc_hi", "delta_c_vs_baseline", "delta_c_lo", "delta_c_hi"):
        if col not in m:
            m[col] = np.nan
    return pd.DataFrame({
        "endpoint": m["endpoint"], "landmark_days": m["landmark_days"], "config": m["config"],
        "n": m["n_patients"], "events": m["n_events"],
        "C": [fmt(*r) for r in m[["c_index", "c_index_lo", "c_index_hi"]].to_numpy()],
        "mean AUC(t)": [fmt(*r) for r in m[["mean_auc", "mean_auc_lo", "mean_auc_hi"]].to_numpy()],
        "Δ C vs age": [fmt(*r) for r in m[["delta_c_vs_baseline", "delta_c_lo", "delta_c_hi"]].to_numpy()],
        "Δ AUC vs age": [fmt(*r) for r in m[["delta_auc_vs_baseline", "delta_auc_lo", "delta_auc_hi"]].to_numpy()],
        "IBS": m["ibs"].round(3),
    })

order = {e: i for i, e in enumerate(ENDPOINTS)}
summary_table(metrics).sort_values(["endpoint", "landmark_days", "config"],
                                   key=lambda s: s.map(order) if s.name == "endpoint" else s).reset_index(drop=True)

### Selected trees

The `all` row is the full-cohort refit, which is the tree one would report. Folds `0`–`4` show how stable the
selection is.

In [ ]:
sel = result["selection"]
sel[["landmark_days", "fold", "config", "max_depth", "min_leaf", "n_leaves", "selection_key",
     "mean_auc_t_cv_mean", "cv_mean", "n_fit", "events_fit", "splits"]].sort_values(["landmark_days", "config", "fold"])

In [ ]:
leaves = result["leaves"]
h0, h1 = CFG.horizons_days
cols = ["landmark_days", "config", "leaf", "rule", "n", "events", "rate", "relative_rate",
        f"S_{h0}d", f"S_{h0}d_lower", f"S_{h0}d_upper", f"S_{h1}d", f"S_{h1}d_lower", f"S_{h1}d_upper"]
leaves.query("fold == 'all'")[cols].sort_values(["landmark_days", "config", "rate"]).round(4)

### The trees

Each selected tree as a text diagram. Children follow rpart numbering: node `2k` is the item **false**
(`NOT(item)`), node `2k+1` the item **true**. Every node shows n, platinum events, the shrunken event rate
(`(d + 1) / (z + 1/root rate)`, per week) and that rate relative to the root. Set `SHOW_FOLD_TREES = True` to also
print the five outer-fold trees.

In [ ]:
SHOW_FOLD_TREES = False   # True also prints the five outer-fold trees (selection stability)

def _tree_nodes(tree):
    """{node id: (split item name or None, n, events, rate)} for a fitted SurvivalTree."""
    return {nid: (None if nd.split is None else tree.items[nd.split].get("name", f"item{nd.split}"),
                  int(nd.n), float(nd.d), tree.leaf_rate(nid)) for nid, nd in tree.nodes.items()}

def _bundle_nodes(model):
    """The same, from a federated bundle model dict (rate = (d + 1) / (z + 1 / root_rate), as leaf_rate)."""
    rr = model["root_rate"]
    return {nd["id"]: (None if nd.get("item") is None else nd["item"].get("name", f"item{nd['split_item']}"),
                       int(nd["n"]), float(nd["events"]), (nd["events"] + 1.0) / (nd["z"] + 1.0 / rr))
            for nd in model["nodes"]}

def print_tree(nodes, title=""):
    """Indented text tree. Children follow rpart numbering: 2k = item false, 2k+1 = item true."""
    root_rate = nodes[1][3]
    if title:
        print(title)
    def label(nid):
        split, n, d, rate = nodes[nid]
        kind = "" if split is not None else "  ◀ leaf"
        return f"[{nid}] n={n:,} events={d:g} rate={rate:.4g} ({rate / root_rate:.2f}× root){kind}"
    def rec(nid, prefix):
        split = nodes[nid][0]
        if split is None:
            return
        kids = [(2 * nid, f"NOT({split})"), (2 * nid + 1, split)]
        for i, (cid, cond) in enumerate(kids):
            last = i == len(kids) - 1
            print(f"{prefix}{'└─' if last else '├─'} {cond} → {label(cid)}")
            rec(cid, prefix + ("   " if last else "│  "))
    print(label(1).replace("[1]", "[1] all patients"))
    rec(1, "")
    print()

for (lm, config), ft in sorted(result["refits"].items()):
    b = ft.best
    print_tree(_tree_nodes(ft.tree),
               f"=== landmark {lm}d · {config} · fold all · depth {int(b['max_depth'])}, "
               f"{len(ft.tree.leaves())} leaves, min_leaf {ft.min_leaf} · fit n={ft.n_fit:,}, events={ft.events_fit:,}")
if SHOW_FOLD_TREES:
    for (lm, config, k), ft in sorted(result["models"].items(), key=lambda kv: (kv[0][0], kv[0][1], kv[0][2])):
        print_tree(_tree_nodes(ft.tree), f"--- landmark {lm}d · {config} · outer fold {k} · "
                                          f"{len(ft.tree.leaves())} leaves · fit n={ft.n_fit:,}")

### Kaplan–Meier by leaf of the full-cohort tree

These plots are **descriptive**. The `all` tree was fit on these platinum rows, so the platinum panels are in sample.
The NEPC and AVPC panels use the same leaves and those endpoints' own outcomes, which the tree never saw. Leaves are
ordered by rate. The legend gives leaf id, rate and n; the leaf table above gives each rule.

In [ ]:
def km_by_leaf(config="both"):
    lms = [lm for lm in CFG.landmarks if (lm, config) in result["refits"]]
    if not lms:
        print(f"no {config} refit"); return
    fig, axes = plt.subplots(len(lms), len(ENDPOINTS), figsize=(4.2 * len(ENDPOINTS), 3.4 * len(lms)),
                             squeeze=False, sharey=True)
    for i, lm in enumerate(lms):
        ft = result["refits"][(lm, config)]
        order_ = sorted(ft.tree.leaves(), key=ft.tree.leaf_rate)
        colors = plt.cm.viridis(np.linspace(0.05, 0.9, len(order_)))
        for j, ep in enumerate(ENDPOINTS):
            ax, wide = axes[i, j], result["frames"].get((ep, lm))
            if wide is None or wide.empty:
                ax.set_axis_off(); continue
            leaf, _ = ft.predict(wide)
            for lf, c in zip(order_, colors):
                m = leaf == lf
                if m.sum() == 0:
                    continue
                KaplanMeierFitter().fit(wide.loc[m, lt.DURATION_COL] / 365.25, wide.loc[m, lt.EVENT_COL],
                                        label=f"leaf {lf}: rate {ft.tree.leaf_rate(lf):.2f}, n={int(m.sum())}"
                                        ).plot_survival_function(ax=ax, ci_show=False, color=c)
            ax.set_title(f"{ep}, landmark {lm}d"); ax.set_xlabel("years from landmark")
            ax.legend(fontsize=6, loc="lower left")
        axes[i, 0].set_ylabel("event-free")
    fig.suptitle(f"{config}: full-cohort platinum tree, leaves applied to each endpoint", y=1.0)
    fig.tight_layout()
    return fig

km_by_leaf("both");

## Federated tree (no DFCI) scored on the same frames

The frozen bundle is scored once with the cross-fit's seed and bootstrap count. The bundle's `baseline` (age-only)
tree is the paired reference. If no bundle has been delivered yet, this section is skipped.

In [ ]:
TRANSFER_KW = dict(endpoints=ENDPOINTS, lab_mapping=lab_mapping, exclusion=CFG.exclusion,
                   landmarks=CFG.landmarks, n_bootstrap=CFG.n_bootstrap, seed=CFG.seed,
                   max_followup_days=CFG.max_followup_days)

fed_res = None
if FEDERATED_TREE_PATHS:
    if len(FEDERATED_TREE_PATHS) > 1:
        print("WARNING: several no-DFCI tree bundles; merging them (a landmark delivered twice stops here)")
    parts = [load_bundle(p) for p in FEDERATED_TREE_PATHS]
    for p, b in zip(FEDERATED_TREE_PATHS, parts):
        assert b.get("model_family") == lt.MODEL_FAMILY, f"{p}: model_family {b.get('model_family')!r}"
        assert "DFCI" not in [str(s).upper() for s in b.get("training_sites") or []], f"{p} was trained on DFCI"
    fed_bundle = merge_bundles(parts, sources=[str(p) for p in FEDERATED_TREE_PATHS]) if len(parts) > 1 else parts[0]
    print(f"training sites: {fed_bundle.get('training_sites')}, label {fed_bundle.get('analysis_label')!r}")
    fed_res = fet.evaluate_transfer(local, {"federated_tree": fed_bundle}, **TRANSFER_KW)
    TRANSFER_DIR.mkdir(parents=True, exist_ok=True)
    for name in ("metrics", "auc_t"):
        fed_res[name].to_csv(TRANSFER_DIR / f"transfer_{name}.csv", index=False)
    fed_res["scores"].to_csv(TRANSFER_DIR / "transfer_scores.csv", index=False)   # patient level
    with open(TRANSFER_DIR / "transfer_run.json", "w") as fh:
        json.dump({"data": str(DATA_PATH), "bundles": [str(p) for p in FEDERATED_TREE_PATHS],
                   "training_sites": fed_bundle.get("training_sites"), **{k: v for k, v in TRANSFER_KW.items()
                   if k != "lab_mapping"}}, fh, indent=2, default=str)
    print(f"wrote {TRANSFER_DIR}")
else:
    print(f"no {FEDERATED_TREE_FILE} in a no_DFCI directory under {RHINO_ROOT}; federated transfer skipped")

In [ ]:
if fed_res is not None:
    rows = []
    for m in fed_bundle["models"]:
        for lf in m["leaves"]:
            rows.append({"landmark_days": m["landmark_days"], "config": m["config"], "leaf": lf["id"],
                         "rule": lf["rule"], "n (training)": lf["n"], "events (training)": lf["events"],
                         "rate": lf["rate"]})
    display(pd.DataFrame(rows).sort_values(["landmark_days", "config", "rate"]))
    for m in sorted(fed_bundle["models"], key=lambda m: (m["landmark_days"], m["config"])):
        print_tree(_bundle_nodes(m), f"=== federated no-DFCI · landmark {m['landmark_days']}d · {m['config']} · "
                                     f"{len(m['leaves'])} leaves")

## Local out-of-fold vs federated transfer

The two analyses use the same DFCI frames and bootstrap seed. The local rows are cross-fitted DFCI trees. The
federated rows are one frozen tree trained without DFCI.

`n` can differ slightly between them. The local rows keep patients with an out-of-fold score for both configs, while
the federated rows keep every patient in the frame.

In [ ]:
tables = [summary_table(metrics).assign(analysis="local cross-fit (DFCI)")]
if fed_res is not None:
    tables.append(summary_table(fed_res["metrics"]).assign(analysis="federated, no DFCI"))
cmp = pd.concat(tables, ignore_index=True)
cmp.pivot_table(index=["endpoint", "landmark_days", "config"], columns="analysis",
                values=["C", "mean AUC(t)", "Δ AUC vs age"], aggfunc="first").reindex(
    sorted({(e, l, c) for e, l, c in cmp[["endpoint", "landmark_days", "config"]].itertuples(index=False)},
           key=lambda k: (order[k[0]], k[1], k[2])))

In [ ]:
fig, axes = plt.subplots(1, len(CFG.landmarks), figsize=(4.5 * len(CFG.landmarks), 3.4), squeeze=False, sharey=True)
both = {"local cross-fit (DFCI)": metrics}
if fed_res is not None:
    both["federated, no DFCI"] = fed_res["metrics"]
for ax, lm in zip(axes[0], CFG.landmarks):
    for k, (name, m) in enumerate(both.items()):
        for c, config in enumerate(lt.CONFIGS):
            z = m[(m["landmark_days"] == lm) & (m["config"] == config)].set_index("endpoint").reindex(ENDPOINTS)
            x = np.arange(len(ENDPOINTS)) + (k * 2 + c - 1.5) * 0.17
            lo = (z["mean_auc"] - z.get("mean_auc_lo", z["mean_auc"])).clip(lower=0)
            hi = (z.get("mean_auc_hi", z["mean_auc"]) - z["mean_auc"]).clip(lower=0)
            ax.errorbar(x, z["mean_auc"], yerr=[lo, hi], fmt="o" if config == "both" else "s",
                        mfc="none" if config == "baseline" else None, color=f"C{k}", capsize=2,
                        label=f"{name}: {config}" if lm == CFG.landmarks[0] else None)
    ax.axhline(0.5, color="grey", lw=0.6, ls=":")
    ax.set_xticks(range(len(ENDPOINTS)), ENDPOINTS); ax.set_title(f"landmark {lm}d")
axes[0, 0].set_ylabel("mean AUC(t), 95% CI")
fig.legend(loc="lower center", ncol=2, fontsize=7, bbox_to_anchor=(0.5, -0.12))
fig.tight_layout();

## Gleason / sequenced subsets: trained outside, applied inside

The same transfer design within DFCI. For each subset and landmark, a tree is selected and fit (inner CV, as above) on
the **platinum** rows (ADT clock) of every DFCI patient **outside** the subset. It is then applied to the subset's
platinum, NEPC and AVPC rows. No subset patient is in the training rows (asserted). Subsets match 04's Figure 5/6 and
checklist-extension cohorts:

| Subset | Applied on | Patients |
|---|---|---|
| `gleason` | ADT clock | Gleason score available before ADT + landmark |
| `somatic` | sequencing-report clock | sequenced |
| `combined` | sequencing-report clock | sequenced with a Gleason score before sequencing + landmark (Fig 6 b set) |

The trees use only lab and age items. Sequencing status only decides who is in the subset; no somatic feature enters a
tree. Training is always on the ADT clock, so the `somatic` and `combined` rows also move the tree to a new time origin, as
04 does when it re-scores the federated bundles on the sequencing clock. When the federated no-DFCI tree is present, it is
scored in the same call on the same subset frames, so the two rows share bootstrap resamples. The full-cohort local
cross-fit is shown for reference. It covers different patients and is not paired with the subset rows. Outputs go to
`OUTPUT_DIR/subset_transfer/`.

In [ ]:
import dataclasses
from survival_common import federated_scoring as fs

# >>> Edit <<<
SUBSET_DIR = OUTPUT_DIR / "subset_transfer"
SUBSETS = {
    # name: (label, clock, membership columns); membership is judged per landmark (Gleason before ADT/sequencing + lm)
    "gleason":  ("Gleason available",      "adt",        (fcc.GLEASON_FEATURE,)),
    "somatic":  ("Sequenced",              "sequencing", (fcc.HAS_SEQUENCING,)),
    "combined": ("Sequenced with Gleason", "sequencing", (fcc.HAS_SEQUENCING, fcc.GLEASON_FEATURE)),
}

clin_gleason, clin_somatic, clin_somatic_features = fcc.load_clinical_sources(
    fcc.sg.DEFAULT_GLEASON_PATH, fcc.sg.DEFAULT_SOMATIC_PATH, fcc.sg.DEFAULT_SOMATIC_MANIFEST_PATH)
specimen = fcc.select_specimen(local, clin_somatic, clin_somatic_features)
local_kept = local
if CFG.exclusion != "none":
    # As in 04: judge the exclusion on the ADT clock, then re-anchor the patients it keeps.
    kept, _ = fs.apply_exclusion(fet.to_federated_long(local, "platinum", lab_mapping)[0], CFG.exclusion)
    local_kept = local[pd.to_numeric(local[fet.LOCAL_ID_COL], errors="coerce").isin(kept[fs.ID_COL].unique())]
local_seq = fcc.sequencing_anchored_local(local_kept, specimen)
clinical = {"adt": fcc.build_clinical_features(local, CFG.landmarks, gleason=clin_gleason),
            "sequencing": fcc.build_clinical_features(local_seq, CFG.landmarks, gleason=clin_gleason,
                                                      specimen=specimen, somatic_features=clin_somatic_features)}
eval_local = {"adt": (local, CFG.exclusion), "sequencing": (local_seq, "none")}   # seq clock: exclusion already applied
local_ids = pd.to_numeric(local[fet.LOCAL_ID_COL], errors="coerce")

def subset_members(clin, cols, lm):
    c = clin[clin["landmark_days"] == lm]
    keep = np.logical_and.reduce([c[col].notna().to_numpy() for col in cols])
    return set(pd.to_numeric(c.loc[keep, fcc.ID_COL]).astype(int))

sub_metrics, sub_auc, sub_scores, sub_leaves, sub_trees = [], [], [], [], {}
for subset, (label, clock, cols) in SUBSETS.items():
    for lm in CFG.landmarks:
        inside = subset_members(clinical[clock], cols, lm)
        if not inside:
            print(f"{subset} landmark {lm}d: empty subset, skipped"); continue
        # Train: platinum rows on the ADT clock of every patient outside the subset (at this landmark).
        cfg_lm = dataclasses.replace(CFG, landmarks=(lm,), endpoints=(lt.TRAINING_ENDPOINT,))
        train_local = local[~local_ids.isin(inside)]
        frames, _ = lt.build_frames(train_local, lab_mapping, cfg_lm, verbose=False)
        plat = frames[(lt.TRAINING_ENDPOINT, lm)]
        fitted, _ = lt.fit_selected(plat, cfg_lm, seed=CFG.seed, tag=f"{subset}|lm{lm}")
        if lt.BOTH not in fitted or lt.BASELINE not in fitted:
            print(f"{subset} landmark {lm}d: no valid tree for {set(lt.CONFIGS) - set(fitted)}, skipped"); continue
        for config, ft in fitted.items():
            sub_trees[(subset, lm, config)] = ft
            sub_leaves += [{"subset": subset, **row} for row in lt.tree_rows(ft, lm, "outside_subset", cfg_lm)[1]]
        bundle = lt.refit_bundle({"refits": {(lm, c): ft for c, ft in fitted.items()}}, cfg_lm,
                                 label=f"dfci_outside_{subset}")
        # Apply: the subset's own rows (its clock) for platinum, NEPC and AVPC. The federated tree, when present,
        # is scored in the same call, so both share frames and bootstrap resamples.
        ev_src, ev_excl = eval_local[clock]
        ev = ev_src[pd.to_numeric(ev_src[fet.LOCAL_ID_COL], errors="coerce").isin(inside)]
        assert set(plat[lt.ID_COL].astype(int)).isdisjoint(inside), "subset patient in the training rows"
        bundles = {"outside_subset": bundle}
        if fed_res is not None:
            bundles["federated_tree"] = fed_bundle
        r = fet.evaluate_transfer(ev, bundles, **{**TRANSFER_KW, "landmarks": (lm,), "exclusion": ev_excl},
                                  verbose=False)
        tag = dict(subset=subset, subset_label=label, clock=clock, n_train=len(plat),
                   events_train=int(plat[lt.EVENT_COL].sum()))
        sub_metrics.append(r["metrics"].assign(**tag)); sub_auc.append(r["auc_t"].assign(**tag))
        sub_scores.append(r["scores"].assign(subset=subset, clock=clock))
        print(f"{subset} landmark {lm}d: trained on {len(plat):,} outside ({tag['events_train']} platinum events), "
              f"applied to {len(inside):,} in subset")

sub_metrics = pd.concat(sub_metrics, ignore_index=True)
SUBSET_DIR.mkdir(parents=True, exist_ok=True)
sub_metrics.to_csv(SUBSET_DIR / "subset_transfer_metrics.csv", index=False)
pd.concat(sub_auc, ignore_index=True).to_csv(SUBSET_DIR / "subset_transfer_auc_t.csv", index=False)
pd.concat(sub_scores, ignore_index=True).to_csv(SUBSET_DIR / "subset_transfer_scores.csv", index=False)  # patient level
pd.DataFrame(sub_leaves).to_csv(SUBSET_DIR / "subset_transfer_leaves.csv", index=False)
print(f"wrote {SUBSET_DIR}")

In [ ]:
ANALYSIS = {"outside_subset": "DFCI outside subset → subset", "federated_tree": "federated no-DFCI → subset"}
tabs = [summary_table(metrics).assign(subset="full cohort", analysis="local cross-fit (DFCI)")]
for b, m in sub_metrics.groupby("bundle"):
    tabs.append(summary_table(m).assign(subset=m["subset"].to_numpy(), analysis=ANALYSIS.get(b, b)))
sub_cmp = pd.concat(tabs, ignore_index=True)
subset_order = {s: i for i, s in enumerate(["full cohort", *SUBSETS])}
sub_cmp.pivot_table(index=["endpoint", "landmark_days", "config"], columns=["subset", "analysis"],
                    values=["n", "events", "mean AUC(t)", "Δ AUC vs age"], aggfunc="first").sort_index(
    axis=1, key=lambda s: s.map(subset_order) if s.name == "subset" else s).reindex(
    sorted({tuple(k) for k in sub_cmp[["endpoint", "landmark_days", "config"]].itertuples(index=False)},
           key=lambda k: (order[k[0]], k[1], k[2])))

In [ ]:
for (subset, lm, config), ft in sub_trees.items():
    b = ft.best
    print_tree(_tree_nodes(ft.tree),
               f"=== {subset} · landmark {lm}d · {config} · trained outside the subset · depth "
               f"{int(b['max_depth'])}, {len(ft.tree.leaves())} leaves, min_leaf {ft.min_leaf} · "
               f"fit n={ft.n_fit:,}, events={ft.events_fit:,}")

## Check: the local refit bundle scores through the transfer wiring

The `all` refits are exported as a federated-schema bundle (`local_survival_tree_model.json`). Scoring it with
`fet.evaluate_transfer` has to reproduce the refit's own leaves. This checks the wiring only. These numbers are in
sample and are **not** a validation.

In [ ]:
local_bundle = load_bundle(LOCAL_DIR / lt.BUNDLE_FILE)
chk = fet.evaluate_transfer(local, {"local_tree": local_bundle}, **{**TRANSFER_KW, "n_bootstrap": 0}, verbose=False)
s = chk["scores"]
for (lm, config), ft in result["refits"].items():
    for ep in ENDPOINTS:
        wide = result["frames"][(ep, lm)]
        leaf, rate = ft.predict(wide)
        got = s[(s["endpoint"] == ep) & (s["landmark_days"] == lm) & (s["config"] == config)].set_index("person_id")
        got = got.loc[wide[lt.ID_COL]]
        assert np.array_equal(got["leaf"].to_numpy(), leaf) and np.allclose(got["risk_score"], rate, rtol=0, atol=0)
print(f"local bundle: {len(local_bundle['models'])} trees score identically through federated_inference")